# Observed gaits - Random Priors
3 dimensions for each strut RPM, where fitness (displacement) is represented by color.

In [11]:
import pandas as pd
import plotly.express as px

# 1. Load the data 
# We use nrows=50 to grab only the actual trials and ignore the summary text at the bottom of the CSV
path = './bo_results_random_priors/bo_results_20260724_170359.csv'
df = pd.read_csv(path, nrows=50)

# 2. Create the interactive 3D scatter plot
fig = px.scatter_3d(
    df, 
    x='RPM_1', 
    y='RPM_2', 
    z='RPM_3',
    color='Displacement_mm',
    size='Displacement_mm',      # Larger dots = further displacement
    size_max=25,                 # Max dot size
    color_continuous_scale='Inferno', # A great colormap for heat (Black -> Purple -> Orange -> Yellow)
    title='SPVVVECTR Tensegrity BO Results (Random Priors)',
    hover_data=['Trial_Number', 'Phase'],
    labels={
        'RPM_1': 'Strut 1 (RPM)',
        'RPM_2': 'Strut 2 (RPM)',
        'RPM_3': 'Strut 3 (RPM)',
        'Displacement_mm': 'Distance (mm)'
    }
)

# 3. Adjust the layout for better viewing
fig.update_layout(
    scene=dict(
        xaxis=dict(range=[-1000, 1000]),
        yaxis=dict(range=[-1000, 1000]),
        zaxis=dict(range=[-1000, 1000])
    ),
    margin=dict(l=0, r=0, b=0, t=40)
)

# 4. Display the graph
fig.show(renderer="browser")

# Surrogate Model - Random Priors
Displays the predicted fitness for every observed/unobserved gait within our parameter space [-1000, 1000]

In [14]:
import numpy as np
import plotly.graph_objects as go
from skopt import load
from skopt.learning import GaussianProcessRegressor
from skopt.learning.gaussian_process.kernels import Matern

# 1. Load your historical physical data
pkl_path = "./bo_results_random_priors/bo_model_20260724_170359.pkl"  
res = load(pkl_path)
X_data = res.x_iters
y_data = res.func_vals # (These are your negated displacements)

# 2. Fix the noise variance scaling trap!
raw_noise_variance = 3799.32
# Since normalize_y=True scales the target variance to 1, we must scale alpha proportionally
scaled_alpha = raw_noise_variance / np.var(y_data)

print(f"Original Alpha: {raw_noise_variance}")
print(f"Corrected Scaled Alpha: {scaled_alpha:.5f}")

# 3. Train a NEW, Corrected Gaussian Process
fixed_gp = GaussianProcessRegressor(
    kernel=Matern(nu=2.5), 
    alpha=scaled_alpha, 
    normalize_y=True
)

# skopt scales the X coordinates internally to [0,1], so we must transform them before fitting
X_transformed = res.space.transform(X_data)
fixed_gp.fit(X_transformed, y_data)

# 4. Create the 3D grid
axis_vals = np.linspace(-1000, 1000, 20)
R1, R2, R3 = np.meshgrid(axis_vals, axis_vals, axis_vals)
grid_points_raw = np.c_[R1.ravel(), R2.ravel(), R3.ravel()]

# 5. Predict using the FIXED model
grid_points_transformed = res.space.transform(grid_points_raw.tolist())
predicted_displacement = -fixed_gp.predict(grid_points_transformed) # Negate back to positive mm

# 6. Build the Volume (Heatmap)
fig = go.Figure(data=go.Volume(
    x=grid_points_raw[:, 0],
    y=grid_points_raw[:, 1],
    z=grid_points_raw[:, 2],
    value=predicted_displacement,
    isomin=np.min(predicted_displacement),
    isomax=np.max(predicted_displacement),
    opacity=0.3,             
    surface_count=20,        
    colorscale='Inferno',
    colorbar=dict(title="Predicted<br>Displacement (mm)")
))

fig.update_layout(
    scene=dict(
        xaxis_title='Strut 1 (RPM)',
        yaxis_title='Strut 2 (RPM)',
        zaxis_title='Strut 3 (RPM)'
    ),
    title="Corrected Gaussian Process: True Displacement Landscape",
    margin=dict(l=0, r=0, b=0, t=40)
)

fig.show(renderer="browser")

Original Alpha: 3799.32
Corrected Scaled Alpha: 0.38246
